# M2b: run the `fusion-eval-v1` auto-label job on a Colab GPU

The job proposes car-part masks (Grounding DINO + SAM 2) on 100 frozen CarDD test photos.
On a laptop CPU it takes about 35 minutes; on a Colab T4 GPU it takes a few minutes.

**Before you start:** on your laptop run `uv run python scripts/make_autolabel_bundle.py`,
which writes `var/colab/autolabel-bundle.zip`.

**In Colab:** Runtime > Change runtime type > **T4 GPU**, then run the cells in order.

## 1. Check the GPU and Python version

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys

print(sys.version)
assert sys.version_info >= (3, 12), "ClaimLens needs Python 3.12 or newer"

## 2. Get the code and install it

Colab already has PyTorch with CUDA; this adds ClaimLens and the model libraries.

In [ ]:
!git clone --depth 1 -b feat/m2b-review-autolabel https://github.com/kb2326/claimlens.git
%cd claimlens
!pip install -q -e . "transformers>=5.0" opencv-python-headless

## 3. Upload the bundle

Choose `autolabel-bundle.zip` from `var/colab/` on your laptop.

In [ ]:
from google.colab import files

uploaded = files.upload()
!unzip -q -o autolabel-bundle.zip -d .
!ls data/interim/damage-v1

## 4. Run the job

The first run downloads about 800 MB of model weights.

In [ ]:
!HF_HUB_DISABLE_SYMLINKS_WARNING=1 claimlens data autolabel fusion-eval-v1

## 5. Download the results

Unzip `autolabel-results.zip` in the repository root on your laptop.

In [ ]:
outputs = "data/interim/fusion-eval-v1 reports/data/fusion-eval-v1-autolabel.json"
!zip -q -r autolabel-results.zip {outputs}
files.download("autolabel-results.zip")